In [1]:
%pip install plotly
import os, warnings
import pandas as pd
import numpy as np

# Conexão com o banco de dados
from sqlalchemy import create_engine

# Aprendizado de máquina
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Visualização - Dashboard
import plotly.graph_objects as go
from plotly.subplots import make_subplots

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Como tirar a chatice do treinamento
warnings.filterwarnings("ignore")

In [9]:
# Conectar ao banco de dados
ENGINE_URL = 'mysql+pymysql://root:@localhost:3306/bolsa_familia'
engine = create_engine(ENGINE_URL, echo=False)

query = '''
        SELECT `MÊS COMPETÊNCIA`, `UF`,
        COUNT(*) AS qtd_parcelas,
        AVG(`VALOR PARCELA`) AS valor_medio,
        SUM(`VALOR PARCELA`) AS valor_total
        FROM bolsa_familia
        GROUP BY `MÊS COMPETÊNCIA`, `UF`
        ORDER BY `MÊS COMPETÊNCIA`, `UF`
        '''
df = pd.read_sql(query, engine)

In [11]:
# EDA
df_uf = df.groupby('UF').agg(
    media_valor=('valor_medio', 'mean'),
    total_parcelas=('qtd_parcelas', 'sum')
).reset_index()

In [16]:
serie = df_uf['media_valor']
q1, q2, q3 = np.percentile(serie, [25, 50, 75])
# Usando método parecido com o que foi usado no notebook 17.banco.ipynb
iqr = q3 - q1
display(f'Média: {serie.mean():.2f}')
display(f'Mediana: {serie.median():.2f}')
display(f'Q1: {q1:.2f}, Q2: {q2:.2f}, Q3: {q3:.2f}')
display(f'Assimetria: {serie.skew():.3f}')
display(f'Curtose: {serie.kurtosis():.3f}')
display(f'IQR: {iqr:.2f}')

'Média: 678.24'

'Mediana: 667.56'

'Q1: 662.09, Q2: 667.56, Q3: 688.32'

'Assimetria: 1.122'

'Curtose: 0.469'

'IQR: 26.22'

In [19]:
# Aprendizado // Agrupamento // Não supervisionado
# Treino e teste de previsão
# No aprendizado não supervisionado fazem-se os agrupamentos sem determinar previamente as classes dos dados
# Vamos dizer o que queremos usar para agrupar os dados
features = df_uf [['media_valor', 'total_parcelas']].values
# Cada coluna vai representar um plano de características para o agrupamento
# Via de regra consegue-se visualizar até 3 dimensões
# Mais colunar vai demandar o uso de algum algoritmo para fazer uma redução para 2D ou 3D
# Boas práticas para aprendizado de máquina
scaler = StandardScaler()
features_norm = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=5, random_state=27, n_init=10)
# Talvez um número inicial próximo a média fosse uma boa escolha
# devido à distribuição dos dados
df_uf['cluster'] = kmeans.fit_predict(features_norm)
display(df_uf.groupby('cluster')[['UF', 'media_valor']].mean())

TypeError: dtype 'str' does not support operation 'mean'